# VisWord — all experiments (merged)

Single Colab runtime for the full experimental suite. Each Part can be run
top-to-bottom; later Parts depend on earlier Parts (Drive + cache state).

## Table of contents

- **Part 0 — Setup** (~5 min, any runtime): clone repo, install deps, mount Drive, pin DINOv2
- **Part 1 — Prefetch** (~1–6 h, CPU OK): download 100k–500k rows of wiki-ss-corpus to Drive
- **Part 2 — Zero-shot vision** (~30 min, T4): rows 1–6 (random / ImageNet / DINO-v1 / DINOv2 / mean-patch / CLIP)
- **Part 3 — Text + multimodal** (~30 min, T4): rows 16–20 (BERT / MiniLM / CLIP-text / cross-modal)
- **Part 4 — Training ladder** (~30–60 min/row, A100/L4 preferred): rows 7–12
- **Part 5 — Interpretability** (~5 min/run, T4): attention / clusters / dustbin / CLS-vs-VLAD maps

## Skip instructions

If you've already run an earlier Part in a prior session, you can **skip straight
to the Setup cell, run that, then jump** to the Part you want — Drive persists all
state (data + runs + HF cache) between sessions.


## Session init — RUN THIS FIRST in every Colab session

Mounts Drive, sets up the repo (from Drive zip, falls back to GitHub if you've pushed one), installs missing deps, pins DINOv2 hub commit.

**Pre-req:** one of
- *(Option A, recommended)* on VALAR run `scripts/package_for_colab.sh`, scp the resulting `VISWORD_src.zip` off, upload it to `MyDrive/VISWORD/VISWORD_src.zip`
- *(Option B)* push the repo to GitHub and edit `REPO_URL` below


In [ ]:
# === Session init ===
import os, sys, zipfile, subprocess
from pathlib import Path

PROJECT  = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
REPO_URL = 'https://github.com/hkanpak21/VISWORD.git'   # edit if using Option B

# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
os.makedirs(PROJECT, exist_ok=True)
for sub in ('data', 'runs', 'hf_cache'):
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)

# 2) Get the source tree into /content/VISWORD
def _looks_ok(root):
    return (Path(root) / 'src' / 'visword' / 'data' / 'prefetch.py').exists()

if not _looks_ok(REPO_DIR):
    zip_path = Path(PROJECT) / 'VISWORD_src.zip'
    drive_src = Path('/content/drive/MyDrive/VISWORD_src')
    if zip_path.exists():
        print(f'extracting {zip_path} -> {REPO_DIR} ...')
        os.makedirs(REPO_DIR, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(REPO_DIR)
    elif drive_src.exists():
        print(f'copying {drive_src} -> {REPO_DIR} ...')
        subprocess.run(['cp', '-r', str(drive_src), REPO_DIR], check=True)
    else:
        print(f'attempting git clone from {REPO_URL} ...')
        rc = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR]).returncode
        if rc != 0 or not _looks_ok(REPO_DIR):
            raise RuntimeError(
                'Could not obtain VISWORD source.\n'
                'Do one of:\n'
                f'  A) Upload VISWORD_src.zip to {zip_path}\n'
                f'  B) Upload unzipped folder to {drive_src}\n'
                '  C) Push repo to GitHub and set REPO_URL above')

assert _looks_ok(REPO_DIR), f'{REPO_DIR} is missing src/visword/data/prefetch.py'
print(f'repo OK at {REPO_DIR}')

# 3) sys.path + working dir
for p in (f'{REPO_DIR}/src', f'{REPO_DIR}/third_party/salad'):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO_DIR)

# 4) Env vars (Drive-persistent HF + torch caches)
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['HF_DATASETS_CACHE'] = f'{PROJECT}/hf_cache/datasets'
os.environ['HF_HUB_CACHE'] = f'{PROJECT}/hf_cache/hub'
os.environ['TORCH_HOME'] = f'{PROJECT}/hf_cache/torch'

# 5) Install missing deps (fast no-op if already installed)
try:
    import transformers, sentence_transformers, open_clip, timm  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers', 'sentence-transformers', 'open-clip-torch',
                    'timm', 'pytorch-metric-learning', 'huggingface_hub', 'pyyaml'],
                   check=True)

# 6) Verify critical import
from visword.data.prefetch import prefetch_wiki_ss  # noqa
print('visword.data.prefetch import OK')

# 7) Pin DINOv2 hub commit (idempotent)
subprocess.run([sys.executable, f'{REPO_DIR}/scripts/ensure_dinov2_hub.py'], check=False)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('=== session init complete ===')

> **Later Parts:** the bootstrap cell in each Part below assumes session init has already run. If you open a fresh Colab runtime, re-run the two cells above first.


---

# Part 1 — Prefetch data


In [ ]:
# Re-establish session state (in case this is a fresh Colab runtime).
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['HF_DATASETS_CACHE'] = f'{PROJECT}/hf_cache/datasets'
os.environ['HF_HUB_CACHE'] = f'{PROJECT}/hf_cache/hub'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hkanpak21/VISWORD.git $REPO_DIR
sys.path.insert(0, f'{REPO_DIR}/src')
%cd $REPO_DIR

## Pick target size

Our VALAR cache has 15k. Target here depends on how much Colab wallclock you can afford:
- 100k rows ≈ 1–2 h on Colab
- 500k rows ≈ 4–8 h (one long session; Drive persists so you can resume)

In [ ]:
TARGET_ROWS = 100_000  # change to 500_000 for full scale-up run

## 1 — wiki-ss-corpus (images + titles + texts)

Uses `visword.data.prefetch.prefetch_wiki_ss` (same code as VALAR). Resume-safe: if you re-run after a timeout, it continues from the last checkpointed row.

In [ ]:
from pathlib import Path
from visword.data.prefetch import prefetch_wiki_ss

cache = Path(os.environ['DATA_DIR']) / 'wiki_ss'
summary = prefetch_wiki_ss(cache, TARGET_ROWS, resume=True)
print('wiki-ss prefetch:', summary)

## 2 — Anchor triplets (small, ~30 MB)

In [ ]:
from visword.data.prefetch import prefetch_anchors
anchors_cache = Path(os.environ['DATA_DIR']) / 'wiki_ss_anchors'
print('anchors prefetch:', prefetch_anchors(anchors_cache))

## 3 — Integrity check

In [ ]:
import json
m = json.load(open(cache / 'manifest.json'))
print('wiki_ss rows:', m['num_rows'])
print('first row keys:', list(m['rows'][0].keys()))
print('sample title:', m['rows'][0]['title'])
print('sample text_path:', m['rows'][0]['text_path'])

from visword.data.manifest import verify_fingerprint
print('fingerprint OK:', verify_fingerprint(cache))

## 4 — Optional: parallel speed-up via split slicing

If `prefetch_wiki_ss` (serial streaming) is too slow, run multiple split-slice workers in parallel processes. Colab's single-VM Python can't truly parallelise CPU download, so this is most useful with `multiprocessing` — skip unless you see <1000 rows/min.

In [ ]:
# Uncomment to try the split-slice path in this notebook (requires restarting the runtime after streaming prefetch).
# from visword.data.prefetch import prefetch_split_slice
# summary = prefetch_split_slice(cache, split_start=100000, split_end=200000, idx_base=1_000_000, target_rows=100_000)
# print(summary)

## Next step

→ `02_zeroshot_vision.ipynb` (zero-shot DINOv2 / CLIP / DINO-v1 / ImageNet-ViT)
→ `03_zeroshot_text_multimodal.ipynb` (BERT / MiniLM / CLIP-text / cross-modal)

Both evaluate on the cache you just downloaded.

---

# Part 2 — Zero-shot vision baselines (rows 1–6)


In [ ]:
# Session bootstrap
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json
from pathlib import Path
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['TORCH_HOME'] = f'{PROJECT}/hf_cache/torch'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hkanpak21/VISWORD.git $REPO_DIR
sys.path.insert(0, f'{REPO_DIR}/src')
sys.path.insert(0, f'{REPO_DIR}/third_party/salad')
%cd $REPO_DIR

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Shared eval pipeline from our repo — reuses the exact Phase-1/2 protocol
from visword.config import Config, DataConfig, EvalConfig, CropperConfig, BackboneConfig, SaladConfig, TrainConfig
from visword.eval_phase1 import compute_recall_at_k, _rebuild_eval_dataset
from visword.eval_phase2 import load_val_triplets, _encode_images, _compute_triplet_recall
from visword.data.light_dataset import default_transform

# Shared config: 4000 eval pages (matches VALAR scale-up eval)
def make_cfg(row_label, num_eval=1000):
    return Config(
        experiment_name=f'visword-{row_label}',
        model_kind='zeroshot',
        data=DataConfig(
            wiki_ss_cache_dir=Path(PROJECT) / 'data' / 'wiki_ss',
            anchors_cache_dir=Path(PROJECT) / 'data' / 'wiki_ss_anchors',
            num_train_samples=0, num_eval_samples=num_eval),
        cropper=CropperConfig(crop_size=224),
        backbone=BackboneConfig(),
        salad=SaladConfig(),
        train=TrainConfig(epochs=0),
        eval=EvalConfig(phase1_max_pages=num_eval, phase2_max_queries=200),
    )

## Shared eval helpers

In [ ]:
import numpy as np

def run_eval(model, row_label, cfg, image_size=224):
    """Run Phase 1 + Phase 2 and write recall JSONs. Model returns L2-normed descriptors."""
    model.eval().to(device)
    run_dir = Path(PROJECT) / 'runs' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_{row_label}'
    run_dir.mkdir(parents=True, exist_ok=True)

    # Phase 1
    eval_ds = _rebuild_eval_dataset(cfg)
    page_embeds, crop_embeds, crop_to_page = [], [], []
    transform = default_transform(image_size)
    # Encode page-level descriptors: one per eval page (page descriptor = mean of its 4 crops' CLS)
    # We'll actually encode each crop and group later for Phase-1 retrieval.
    import torch.utils.data as D
    for i in range(len(eval_ds)):
        imgs = eval_ds[i]    # returns (k_per_page, 3, H, W) already transformed
        with torch.no_grad():
            d = model(imgs.to(device))
            d = F.normalize(d, p=2, dim=-1)
        crop_embeds.append(d.cpu())
        crop_to_page.extend([i] * d.shape[0])
    crop_embeds = torch.cat(crop_embeds)
    page_embeds = torch.stack([crop_embeds[torch.tensor(crop_to_page) == i].mean(0) for i in range(len(eval_ds))])
    page_embeds = F.normalize(page_embeds, p=2, dim=-1)

    p1 = compute_recall_at_k(
        query_embeds=crop_embeds, gallery_embeds=page_embeds,
        query_to_gallery=torch.tensor(crop_to_page),
        k_values=cfg.eval.k_values)

    # Phase 2 — anchors
    triplets = load_val_triplets(cfg.data.anchors_cache_dir)[:cfg.eval.phase2_max_queries]
    p2_scores = {k: 0.0 for k in cfg.eval.k_values}
    anchor_images = cfg.data.anchors_cache_dir / 'images'
    same_sim, diff_sim, n_valid = [], [], 0
    for t in triplets:
        pos = [anchor_images / p for p in t['positives'] if (anchor_images / p).exists()]
        neg = [anchor_images / p for p in t['negatives'] if (anchor_images / p).exists()]
        anc = anchor_images / t['anchor']
        if not pos or not anc.exists() or not neg: continue
        n_valid += 1
        ae = _encode_images(model, [anc], transform=transform, target_size=image_size, device=device)
        pe = _encode_images(model, pos + neg, transform=transform, target_size=image_size, device=device)
        sim = (ae @ pe.T).squeeze(0)
        for k in cfg.eval.k_values:
            topk = sim.topk(min(k, len(sim))).indices
            if any(i < len(pos) for i in topk.tolist()): p2_scores[k] += 1
        same_sim.extend(sim[:len(pos)].tolist())
        diff_sim.extend(sim[len(pos):].tolist())
    for k in cfg.eval.k_values:
        p2_scores[k] /= max(n_valid, 1)

    # Write
    json.dump({
        'checkpoint': None, 'checkpoint_step': None,
        'num_pages_evaluated': len(eval_ds),
        'num_crops': len(crop_embeds),
        'recall': {str(k): float(v) for k, v in p1.items()},
        'sanity': {'same_page_sim_mean': 0.0, 'diff_page_sim_mean': 0.0, 'gap': 0.0, 'monotonic': True}
    }, open(run_dir / 'phase1_recall.json', 'w'), indent=2)
    json.dump({
        'checkpoint': None, 'checkpoint_step': None,
        'num_triplets': n_valid,
        'recall': {str(k): v for k, v in p2_scores.items()},
        'sanity': {
            'same_page_sim_mean': float(np.mean(same_sim)) if same_sim else 0,
            'diff_page_sim_mean': float(np.mean(diff_sim)) if diff_sim else 0,
            'gap': float(np.mean(same_sim) - np.mean(diff_sim)) if same_sim and diff_sim else 0,
        }
    }, open(run_dir / 'phase2_recall.json', 'w'), indent=2)
    print(f'{row_label}: P1 R@10={p1[10]:.3f}  P2 R@1={p2_scores[1]:.3f}  (n={n_valid} triplets)')
    return run_dir

## Row 1 — Random-init ViT (sanity floor)

In [ ]:
import timm
class RandomCLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
    def forward(self, x):
        return F.normalize(self.vit(x), p=2, dim=-1)
run_eval(RandomCLS(), 'row01_random_vit', make_cfg('row01_random_vit'))

## Row 2 — ImageNet-supervised ViT

In [ ]:
class ImageNetCLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True, num_classes=0)
    def forward(self, x):
        return F.normalize(self.vit(x), p=2, dim=-1)
run_eval(ImageNetCLS(), 'row02_imagenet_vit', make_cfg('row02_imagenet_vit'))

## Row 3 — DINO v1

In [ ]:
class DINOv1CLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16')
    def forward(self, x):
        return F.normalize(self.vit(x), p=2, dim=-1)
run_eval(DINOv1CLS(), 'row03_dino_v1', make_cfg('row03_dino_v1'))

## Row 4 — DINOv2 zero-shot CLS (matches VALAR row 4)

In [ ]:
class DINOv2CLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    def forward(self, x):
        return F.normalize(self.vit(x), p=2, dim=-1)
run_eval(DINOv2CLS(), 'row04_dinov2_zeroshot', make_cfg('row04_dinov2_zeroshot'))

## Row 5 — DINOv2 zero-shot mean-patch

In [ ]:
class DINOv2MeanPatch(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    def forward(self, x):
        tokens = self.vit.forward_features(x)['x_norm_patchtokens']  # (B, N, 768)
        return F.normalize(tokens.mean(1), p=2, dim=-1)
run_eval(DINOv2MeanPatch(), 'row05_dinov2_mean', make_cfg('row05_dinov2_mean'))

## Row 6 — CLIP image branch

In [ ]:
import open_clip
class CLIPImage(nn.Module):
    def __init__(self):
        super().__init__()
        model, _, _ = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
        self.model = model
    def forward(self, x):
        with torch.no_grad():
            return F.normalize(self.model.encode_image(x).float(), p=2, dim=-1)
run_eval(CLIPImage(), 'row06_clip_image', make_cfg('row06_clip_image'))

## Summary table

In [ ]:
import pandas as pd
rows = []
for d in sorted(Path(f'{PROJECT}/runs').glob('*_row0[1-6]_*')):
    p1 = json.load(open(d / 'phase1_recall.json'))
    p2 = json.load(open(d / 'phase2_recall.json'))
    rows.append({
        'row': d.name.split('_row')[1][:2],
        'label': d.name.split('_', 2)[-1],
        'P1_R@1': p1['recall']['1'],
        'P1_R@10': p1['recall']['10'],
        'P2_R@1': p2['recall']['1'],
        'P2_n_triplets': p2['num_triplets'],
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(f'{PROJECT}/runs/zeroshot_vision_summary.csv', index=False)

---

# Part 3 — Text + multimodal baselines (rows 16–20)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json
from pathlib import Path
from datetime import datetime
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hkanpak21/VISWORD.git $REPO_DIR
sys.path.insert(0, f'{REPO_DIR}/src')
%cd $REPO_DIR
import torch, torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load anchor metadata and triplets

In [ ]:
anchors_root = Path(PROJECT) / 'data' / 'wiki_ss_anchors'
meta = {}
for line in open(anchors_root / 'metadata.jsonl'):
    r = json.loads(line)
    meta[r['image_path']] = r
print('anchor metadata entries:', len(meta))

triplets = [json.loads(l) for l in open(anchors_root / 'triplets_val.jsonl').read().splitlines() if l.strip()]
print('val triplets:', len(triplets))

def text_for(image_fname, mode='title'):
    m = meta.get(image_fname)
    if m is None: return ''
    title = m.get('title', '').replace('_', ' ')
    if mode == 'title': return title
    if mode == 'title_body': return (title + '\n' + (m.get('visible_text_snippet', '') or ''))[:2000]
    return title

## Shared Phase-2 text-retrieval helper

In [ ]:
import numpy as np

def phase2_text_retrieval(encode_texts, row_label, mode='title', k_values=(1,5,10,20), max_triplets=200):
    run_dir = Path(PROJECT) / 'runs' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_{row_label}'
    run_dir.mkdir(parents=True, exist_ok=True)
    scores = {k: 0 for k in k_values}
    same_sim, diff_sim, n_valid = [], [], 0
    for t in triplets[:max_triplets]:
        pos, neg = t['positives'], t['negatives']
        anc = t['anchor']
        if not pos or not neg: continue
        # skip if anchor image is actually missing (upstream HF only ships ~10k)
        if not (anchors_root / 'images' / anc).exists(): continue
        a_text = text_for(anc, mode)
        pool_texts = [text_for(p, mode) for p in pos + neg]
        if not a_text or not any(pool_texts): continue
        n_valid += 1
        a_emb = encode_texts([a_text])
        p_emb = encode_texts(pool_texts)
        sim = (a_emb @ p_emb.T).squeeze(0)
        for k in k_values:
            topk = sim.topk(min(k, len(sim))).indices
            if any(i < len(pos) for i in topk.tolist()): scores[k] += 1
        same_sim.extend(sim[:len(pos)].tolist())
        diff_sim.extend(sim[len(pos):].tolist())
    for k in k_values: scores[k] /= max(n_valid, 1)
    out = {
        'checkpoint': None, 'checkpoint_step': None,
        'num_triplets': n_valid,
        'text_mode': mode,
        'recall': {str(k): scores[k] for k in k_values},
        'sanity': {
            'same_sim_mean': float(np.mean(same_sim)) if same_sim else 0,
            'diff_sim_mean': float(np.mean(diff_sim)) if diff_sim else 0,
            'gap': float(np.mean(same_sim)-np.mean(diff_sim)) if same_sim and diff_sim else 0,
        }
    }
    json.dump(out, open(run_dir / 'phase2_recall.json', 'w'), indent=2)
    print(f'{row_label}: P2 R@1={scores[1]:.3f}  R@5={scores[5]:.3f}  gap={out["sanity"]["gap"]:+.3f}  (n={n_valid})')
    return run_dir, out

## Row 18 — sentence-MiniLM (fastest, run first)

In [ ]:
from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2').to(device)
def minilm_encode(texts):
    with torch.no_grad():
        e = minilm.encode(texts, convert_to_tensor=True, device=device)
    return F.normalize(e, dim=-1).cpu()
phase2_text_retrieval(minilm_encode, 'row18_minilm_title', mode='title')

## Row 16 — BERT-base mean-pooled, title only

In [ ]:
from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained('bert-base-uncased')
bert = AutoModel.from_pretrained('bert-base-uncased').to(device).eval()
def bert_encode(texts, max_length=64):
    enc = tok(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
    with torch.no_grad():
        out = bert(**enc).last_hidden_state  # (B, T, 768)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp_min(1)
    return F.normalize(pooled, dim=-1).cpu()
phase2_text_retrieval(bert_encode, 'row16_bert_title', mode='title')

## Row 17 — BERT with title + body[:200]

In [ ]:
def bert_encode_long(texts): return bert_encode(texts, max_length=256)
phase2_text_retrieval(bert_encode_long, 'row17_bert_title_body', mode='title_body')

## Row 19 — CLIP text branch

In [ ]:
import open_clip
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
clip_model = clip_model.to(device).eval()
clip_tok = open_clip.get_tokenizer('ViT-B-16')
def clip_text_encode(texts):
    tokens = clip_tok(texts).to(device)
    with torch.no_grad():
        e = clip_model.encode_text(tokens).float()
    return F.normalize(e, dim=-1).cpu()
phase2_text_retrieval(clip_text_encode, 'row19_clip_text', mode='title')

## Row 20 — CLIP cross-modal (image query → text pool and vice versa)

Special protocol: anchor is an **image**, pool items are **text descriptions**. This is CLIP's designed use-case. Separately we also run image→image (matching row 6).

In [ ]:
from PIL import Image
def clip_image_encode(image_paths):
    imgs = [clip_preprocess(Image.open(p).convert('RGB')) for p in image_paths]
    x = torch.stack(imgs).to(device)
    with torch.no_grad():
        return F.normalize(clip_model.encode_image(x).float(), dim=-1).cpu()

run_dir = Path(PROJECT) / 'runs' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_row20_clip_crossmodal'
run_dir.mkdir(parents=True, exist_ok=True)
scores = {k: 0 for k in (1, 5, 10, 20)}; n_valid = 0
for t in triplets[:200]:
    pos, neg, anc = t['positives'], t['negatives'], t['anchor']
    if not pos or not neg or not (anchors_root / 'images' / anc).exists(): continue
    valid_pool = [p for p in pos + neg if (anchors_root / 'images' / p).exists()]
    if len(valid_pool) < 2: continue
    n_valid += 1
    a_img = clip_image_encode([anchors_root / 'images' / anc])
    pool_texts = [text_for(p, 'title') for p in valid_pool]
    p_txt = clip_text_encode(pool_texts)
    sim = (a_img @ p_txt.T).squeeze(0)
    n_pos_in_pool = sum(1 for p in pos if p in valid_pool)
    for k in scores:
        topk = sim.topk(min(k, len(sim))).indices
        if any(i < n_pos_in_pool for i in topk.tolist()): scores[k] += 1
for k in scores: scores[k] /= max(n_valid, 1)
json.dump({'num_triplets': n_valid, 'recall': {str(k): v for k, v in scores.items()}, 'protocol': 'image_query_text_pool'},
          open(run_dir / 'phase2_recall.json', 'w'), indent=2)
print(f'row20_clip_crossmodal: P2 R@1={scores[1]:.3f}  R@10={scores[10]:.3f}  (n={n_valid})')

## Summary table (rows 16-20)

In [ ]:
import pandas as pd
rows_out = []
for d in sorted(Path(f'{PROJECT}/runs').glob('*_row1[6-9]_*')) + list(Path(f'{PROJECT}/runs').glob('*_row20_*')):
    r = json.load(open(d / 'phase2_recall.json'))
    rows_out.append({
        'row': d.name.split('_row')[1][:2],
        'label': d.name.split('_', 2)[-1],
        'P2_R@1': r['recall']['1'],
        'P2_R@10': r['recall']['10'],
        'n': r['num_triplets'],
    })
df = pd.DataFrame(rows_out)
print(df.to_string(index=False))
df.to_csv(f'{PROJECT}/runs/zeroshot_text_summary.csv', index=False)

---

# Part 4 — Training ladder (rows 7–15)


In [ ]:
# Session bootstrap
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, yaml, subprocess
from pathlib import Path
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['TORCH_HOME'] = f'{PROJECT}/hf_cache/torch'
os.environ['RUNS_DIR'] = f'{PROJECT}/runs'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hkanpak21/VISWORD.git $REPO_DIR
sys.path.insert(0, f'{REPO_DIR}/src')
sys.path.insert(0, f'{REPO_DIR}/third_party/salad')
%cd $REPO_DIR
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Pick training scale

In [ ]:
import json
cache = Path(os.environ['DATA_DIR']) / 'wiki_ss'
manifest = json.load(open(cache / 'manifest.json'))
print('cache has', manifest['num_rows'], 'rows')

NUM_TRAIN = min(manifest['num_rows'] - 1000, 50_000)   # leave 1000 for eval
NUM_EVAL = 1000
EPOCHS = 3
BATCH_SIZE = 16       # 32 OOMs on T4 for SALAD; A100 can handle 64+
print(f'will train on {NUM_TRAIN} pages × {EPOCHS} epochs, eval on {NUM_EVAL}, bs={BATCH_SIZE}')

## Helper — write a ladder config YAML + invoke `visword.train:main`

In [ ]:
def make_config(row_id, label, model_kind, salad_ablation='full', num_trainable_blocks=4, loss='multisim'):
    cfg = {
        'experiment_name': f'visword-{row_id}-{label}',
        'model_kind': model_kind,
        'data': {'num_train_samples': NUM_TRAIN, 'num_eval_samples': NUM_EVAL},
        'backbone': {'num_trainable_blocks': num_trainable_blocks},
        'salad': {'ablation': salad_ablation},
        'train': {
            'loss': loss, 'k_per_page': 4, 'batch_size': BATCH_SIZE,
            'epochs': EPOCHS, 'lr_backbone': 1e-5, 'lr_head': 5e-4,
            'eval_every_steps': 250, 'num_diag_batches': 3,
        },
        'eval': {'k_values': [1,5,10,20], 'phase1_max_pages': NUM_EVAL, 'phase2_max_queries': 200},
    }
    cfg_path = Path('configs') / f'colab_{row_id}_{label}.yaml'
    yaml.safe_dump(cfg, open(cfg_path, 'w'))
    return cfg_path

def run_training(row_id, label, **kwargs):
    cfg_path = make_config(row_id, label, **kwargs)
    run_name = f'{row_id}_{label}'
    cmd = ['python', '-u', '-m', 'visword.train', '--config', str(cfg_path), '--run-name', run_name]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True, env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'})
    # eval follows
    run_dirs = sorted(Path(PROJECT, 'runs').glob(f'*_{run_name}_*'))
    run_dir = run_dirs[-1]
    subprocess.run(['python', '-u', '-m', 'visword.eval_phase1', '--run-dir', str(run_dir)],
                   check=True, env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'})
    subprocess.run(['python', '-u', '-m', 'visword.eval_phase2', '--run-dir', str(run_dir)],
                   check=True, env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'})
    return run_dir

## Row 9 — DINOv2 + MLP head, last 4 blocks (the CLS baseline)

In [ ]:
run_training('row09', 'cls_main', model_kind='cls', num_trainable_blocks=4, loss='multisim')

## Row 12 — DINOv2 + SALAD-full (the headline SALAD run)

In [ ]:
run_training('row12', 'salad_main', model_kind='salad', salad_ablation='full', loss='multisim')

## Row 7 — Linear probe

In [ ]:
# Linear probe = frozen backbone (num_trainable_blocks=0) + single linear head.
# Our current config has MLP head; for a true linear probe you'd need to patch
# DINOv2CLS.head → nn.Linear. Here we approximate with frozen-backbone MLP.
run_training('row07', 'linear_probe', model_kind='cls', num_trainable_blocks=0, loss='multisim')

## Row 8 — Last 2 blocks fine-tune

In [ ]:
run_training('row08', 'cls_last2', model_kind='cls', num_trainable_blocks=2, loss='multisim')

## Row 10 — InfoNCE loss

In [ ]:
run_training('row10', 'cls_infonce', model_kind='cls', num_trainable_blocks=4, loss='infonce')

## Row 11 — Triplet loss

In [ ]:
run_training('row11', 'cls_triplet', model_kind='cls', num_trainable_blocks=4, loss='triplet')

## Rows 22–23 — CLIP image backbone + MLP / SALAD heads (optional, requires a backbone refactor)

These need a new `CLIPImageBackbone` wrapper that replaces `OfficialDINOv2` in `dinov2_cls.py` / `dinov2_salad.py`. Below is a minimal sketch that subclasses our `DINOv2CLS` / `DINOv2SALAD` but swaps the backbone.

Skip if time-constrained — DINOv2 rows are the more critical experiments.

In [ ]:
# Minimal CLIP-image + SALAD sketch (training implemented via a quick script).
# If you want to run it: write a tiny train_clip_salad.py under repo root that
# replaces `self.backbone = OfficialDINOv2(...)` with `self.backbone = CLIPImageBackbone(...)`.
print('CLIP fine-tune rows (22-24) require a small backbone refactor; see plan Part D.')

## Final summary of trained rows

In [ ]:
import pandas as pd
rows_out = []
for d in sorted(Path(f'{PROJECT}/runs').glob('*_row0[7-9]_*')) + \
         list(Path(f'{PROJECT}/runs').glob('*_row1[0-2]_*')):
    try:
        p1 = json.load(open(d / 'phase1_recall.json'))
        p2 = json.load(open(d / 'phase2_recall.json'))
        rows_out.append({
            'row': d.name.split('_row')[1][:2],
            'label': d.name.split('_', 2)[-1],
            'P1_R@10': p1['recall']['10'],
            'P2_R@1': p2['recall']['1'],
        })
    except Exception:
        pass
df = pd.DataFrame(rows_out)
print(df.to_string(index=False))
df.to_csv(f'{PROJECT}/runs/training_ladder_summary.csv', index=False)

---

# Part 5 — Interpretability sweep


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
from pathlib import Path
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['TORCH_HOME'] = f'{PROJECT}/hf_cache/torch'
sys.path.insert(0, f'{REPO_DIR}/src')
sys.path.insert(0, f'{REPO_DIR}/third_party/salad')
%cd $REPO_DIR

## Discover run dirs that have checkpoints

In [ ]:
runs_root = Path(PROJECT) / 'runs'
trained_runs = sorted([d for d in runs_root.iterdir() if (d / 'checkpoints' / 'best_phase1.pt').exists()])
print(f'{len(trained_runs)} trained runs found:')
for d in trained_runs: print(' ', d.name)

## Invoke `visword.interpret` on each

In [ ]:
for d in trained_runs:
    if (d / 'interpret' / 'attention_sample0.png').exists():
        print(f'skip {d.name} (already has interpret dir)')
        continue
    print(f'\n=== {d.name} ===')
    subprocess.run(
        ['python', '-u', '-m', 'visword.interpret', '--run-dir', str(d), '--k', '4'],
        check=False,
        env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'},
    )

## Display a few artefacts inline

In [ ]:
from IPython.display import Image, display, Markdown
for d in trained_runs:
    interp = d / 'interpret'
    if not interp.exists(): continue
    display(Markdown(f'### {d.name}'))
    for png in sorted(interp.glob('*.png'))[:6]:
        display(Image(str(png), width=400))